In [ ]:
import os
import sys

project_root = '/home/jovyan/project_10x'

sys.path.append(os.path.join(project_root, 'src'))

from utils import *
from sentiment_extraction_MB_10q import *

import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
clean_mem()

In [ ]:
sys.version

# Model - ModernBERT

In [ ]:
model_id = "/home/jovyan/models-2/ModernBERT-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id,
#                                           max_length=length,
                                          padding=True,
                                          truncation=True,
                                          cache_dir="hf_cache")
if not tokenizer.pad_token:
    print("Adding pad token")
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "left"

model = AutoModelForMaskedLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, cache_dir="hf_cache"
).to(device)

config = AutoConfig.from_pretrained(model_id)
length = config.max_position_embeddings
length

In [ ]:
model.name = 'ModernBERT'

In [ ]:
def get_prompt(item_text: str, ending: str) -> str:
    return item_text + "\n\n" + ending 

## 10 Qs

In [ ]:
# import zipfile

# with zipfile.ZipFile('/home/jovyan/datavol-2/files_10qs.zip', 'r') as zip_ref:
#     zip_ref.extractall('/home/jovyan/datavol-2/')

In [ ]:
report_path = '/home/jovyan/datavol-2/'

reports = pd.read_csv(os.path.join(report_path, '10q_filtered.tsv.gz'), sep='\t')
reports = reports[['CIK', 'FILING_DATE', 'ACC_NUM']]
reports['CIK'] = reports['CIK'].astype(str)
reports['FILING_DATE'] = reports['FILING_DATE'].astype(str)
reports['ACC_NUM'] = reports['ACC_NUM'].astype(str)
# reports

In [ ]:
reports['FILING_DATE'].astype(str).str[:4].value_counts().sort_index()

In [ ]:
res_df = reports[reports['FILING_DATE'].astype(str).str[:4] > '2003'].copy().reset_index(drop=True)
res_df = res_df.drop_duplicates().reset_index(drop=True)

res_df.shape, res_df.drop_duplicates().shape

In [ ]:
res_df.info()

In [ ]:
res_df

In [ ]:
res_df.to_csv('/home/jovyan/datavol-2/res_df_10qs_2004_2023.csv', index=True)

In [ ]:
!ls -la /home/jovyan/datavol-2/files_10qs | head -n 5

In [ ]:
items_path = '/home/jovyan/datavol-2/files_10qs'

dataset = Dataset10q(res_df, items_path)
len(dataset)

In [ ]:
ending = "In one word, we are [MASK] regarding the future growth of our company."
inputs = tokenizer(ending, return_tensors="pt", padding=True)
inputs['input_ids'].shape

### Number of symbols

In [ ]:
q99 = 736659
q95 = 377225

chunk_size = length - 100 # for the ending
chunk_size

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer=tokenizer,
                                                                     chunk_size=chunk_size,
                                                                     chunk_overlap=240)

## Growth Sentiment Extraction

### In one word, we are [MASK] regarding the future growth of our company

In [ ]:
ending = "In one word, we are [MASK] regarding the future growth of our company."

In [ ]:
idx = 23

_, item_text = dataset[idx]
date = res_df.iloc[idx].FILING_DATE

item_text = item_text[:20_000]

print(f"{idx} length: {len(item_text)}, date: {date}")

In [ ]:
prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device)
probs

In [ ]:
clean_mem()

In [ ]:
@dataclass(frozen=True)
class Prompt_Strategy:
    name: str
    verbalizer: dict
    prompt: Callable
    top_p: float = 0.9

sentiment_verb = {
    "positive": set(['optimistic', 'confident', 'positive', 'encouraged', 'excited',
                     'enthusiastic', 'hopeful', 'pleased', 'encouraging', 'ambitious',
                     'favorable', 'assured', 'strong', 'good', 'excellent', 'outstanding', 
                     'healthy', 'awesome', 'great', 'fantastic', 'stable',
                     'perfect', 'solid', 'profitable', 'impressive', 'reliable',
                     'thriving', 'optimistic', 'sustainable',
                     'exceptional', 'promising', 'bright', 'attractive'
                     ]),
    
    "negative": set(['cautious', 'concerned', 'pessimistic', 'negative',
                     'uncomfortable', 'uncertain', 'unsure',
                     'skeptical', 'worried', 'worrying', 'discouraged', 
                     'confused', 'doubtful', 'unsatisfied', 'disappointed',
                     'bad', 'poor', 'terrible', 'risky', 'weak', 'dependent',
                     'unstable', 'unhealthy', 'questionable', 'suffering', 'stressed',
                     'unsustainable', 'awful', 'vulnerable',
                     'mediocre', 'horrible', 'precarious', 'declining', 'worsening',
                     'difficult', 'limited', 'challenged', 'disappointed', 'discouraged',
                     'frustrated', 'dissatisfied', 'worried', 'anxious', 'nervous', 'uneasy',
                     'doubtful', 'skeptical', 'apprehensive',
                    ])
}


sentiment_strategy = Prompt_Strategy('sentiment', sentiment_verb, get_prompt)

len(sentiment_verb['positive']), len(sentiment_verb['negative'])

In [ ]:
clean_mem()

In [ ]:
ending = "In one word, we are [MASK] regarding the future growth of our company."

In [ ]:
results = []

df = pd.read_csv('/home/jovyan/datavol-2/sentiments/growth_MB_10qs/results_20260319_182658_1000_reports.csv')

In [ ]:
# df = df.drop('index', axis=1)
# df['level_0'] = df['level_0'].astype(int)
df = df.set_index('index')
results.append(df)
# df

In [ ]:
max_processed_idx = int(max([df.index.max() for df in results if not df.empty]))
max_processed_idx

In [ ]:
batch_size = 2
items_path = '/home/jovyan/datavol-2/files_10qs'

dataset = Dataset10q(res_df, items_path)
print('dataset', len(dataset))

In [ ]:
sampler = IndexBasedSampler(dataset, start_idx=max_processed_idx)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    sampler=sampler,
    collate_fn=split_collator,
    shuffle=False
)

print('dataloader', len(dataloader))

In [ ]:
stats_sent = gather_stats(sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
                          data=dataloader, ending=ending, verbose=True, text_length=q95, 
                          save_path="/home/jovyan/datavol-2/sentiments/growth_MB_10qs/",
                          save_interval=1000, resume=True, max_retries=3, device=device)

In [ ]:
clean_mem()

In [ ]:
time.sleep(60)
clean_mem()
torch.cuda.empty_cache()
time.sleep(60)

In [ ]:
stats_sent.to_csv('/home/jovyan/datavol-2/sentiments/growth_MB_10qs/results_growth_MB_10qs_2004_2023.csv')

In [ ]:
stats_sent = pd.read_csv('/home/jovyan/datavol-2/sentiments/growth_MB_10qs/results_growth_MB_10qs_2004_2023.csv')

In [ ]:
stats_sent['polarity'].hist(bins=100)